# Offline Learning on Different Replay Buffer Compositions

In [15]:
import torch
import numpy as np
from omegaconf import OmegaConf
from functools import partial
import gymnasium as gym
import matplotlib.pyplot as plt
import re
from pathlib import Path
from tqdm.notebook import tqdm

import time
import datetime
import os

import bbrl_utils
from bbrl_utils.notebook import setup_tensorboard
from bbrl.stats import WelchTTest
from bbrl.agents import Agent, Agents, TemporalAgent
from bbrl.agents.gymnasium import ParallelGymAgent, make_env
from bbrl.workspace import Workspace
from bbrl.utils.replay_buffer import ReplayBuffer

import bbrl_gymnasium

from pmind.algorithms import DQN, DDPG, TD3, OfflineTD3
from pmind.losses import dqn_compute_critic_loss, ddqn_compute_critic_loss
from pmind.training import (
    run_dqn,
    run_ddpg,
    run_td3,
)
from pmind.replay import (
    collect_policy_transitions,
    collect_uniform_transitions,
    mix_transitions,
    test_rb_uniform_proportions,
    test_rb_noise_levels
)

from pmind.visualization import plot_perf_vs_rb_composition_from_dict

from pmind.config.loader import load_config

bbrl_utils.setup()

OUTPUT_DIR = f"../results/test_rb_compositions-{datetime.datetime.now().strftime('%Y-%m-%d-%H%M%S')}"
os.mkdir(OUTPUT_DIR)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
ENV_NAMES = (
    "CartPoleContinuous-v1",
    "Pendulum-v1",
    "MountainCarContinuous-v0",
    "LunarLanderContinuous-v3",
)
BUFFER_SIZE = 200_000
BEST_ONLY = True
TEST_RB_COMPOSITIONS = True
TEST_NOISE_LEVELS = True
PROPORTIONS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
SEEDS = [0, 1, 2, 3, 4]
MAX_STEPS = 100#100_000#15_000
NB_EVAL_ENVS = 3#10
EVAL_INTERVAL = 10 #100
BATCH_SIZE = 10#64
NN_ARCHITECTURE = [400,300]

In [20]:
for ENV_NAME in ["CartPoleContinuous-v1"]:#ENV_NAMES:
    print(f"Testing {ENV_NAME}:")
    cfg = load_config("td3")
    cfg = OmegaConf.create(cfg.environments[ENV_NAME])

    best_reward, _ = torch.load(
        f"../models/{ENV_NAME}/best-policy.pt", weights_only=False
    )

    # Load replay buffers from file
    rb_exploit = torch.load(f"../models/{ENV_NAME}/rb-exploit.pt", weights_only=False)
    rb_unif = torch.load(f"../models/{ENV_NAME}/rb-unif.pt", weights_only=False)

    cfg_offline = OmegaConf.create(cfg)

    cfg_offline.algorithm.n_steps = MAX_STEPS
    cfg_offline.algorithm.max_epochs = None

    cfg_offline.algorithm.batch_size = BATCH_SIZE
    cfg_offline.algorithm.architecture.actor_hidden_size = NN_ARCHITECTURE
    cfg_offline.algorithm.architecture.critic_hidden_size = NN_ARCHITECTURE
    

    cfg_offline.algorithm.eval_interval = EVAL_INTERVAL
    cfg_offline.algorithm.nb_evals = NB_EVAL_ENVS  # nb of evaluation envs in parallel

    # learning starts immediately for offline:
    cfg_offline.algorithm.learning_starts = None

    # there is no exploration in offline learning
    cfg_offline.algorithm.action_noise = None
    cfg_offline.algorithm.target_policy_noise = None
    
    if TEST_RB_COMPOSITIONS:
        print("TESTING UNIFORM PROPORTIONS")
        for reward, rb_by_noise in sorted(rb_exploit.items(), reverse=True):
            print(f"Policy with reward {best_reward} :")
            performances = test_rb_uniform_proportions(
                rb_unif,
                rb_by_noise[0],
                BUFFER_SIZE,
                PROPORTIONS,
                TD3,
                cfg_offline,
                SEEDS,
            )
            torch.save(performances, f"{OUTPUT_DIR}/uniform_proportions-{ENV_NAME}-scoring-{reward:.0f}")
            
            if BEST_ONLY:
                print("Skipped intermediate policies")
                break
    
    if TEST_NOISE_LEVELS:
        print("TESTING NOISE LEVELS")
        for reward, rb_by_noise in sorted(rb_exploit.items(), reverse=True):
            print(f"Policy with reward {best_reward} :")
            performances = test_rb_noise_levels(
                rb_by_noise,
                BUFFER_SIZE,
                TD3,
                cfg_offline,
                SEEDS,
            ) 
            torch.save(performances, f"{OUTPUT_DIR}/noise_levels-{ENV_NAME}-scoring-{reward:.0f}")
            
            if BEST_ONLY:
                print("Skipped intermediate policies")
                break


Testing CartPoleContinuous-v1:
TESTING UNIFORM PROPORTIONS
Policy with reward 500.0 :


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Skipped intermediate policies
